![Built with AI](https://img.shields.io/badge/Built%20with-AI-blue.svg)
 [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DS-UdeA/2026-1/blob/main/clases/clase_06/notebooks/bloque2_hash_colisiones.ipynb)

# Bloque 2 — Función Hash y Colisiones

### Estructuras de Datos y Laboratorio — Universidad de Antioquia
**Curso:** Estructuras de Datos y Persistencia · Ingeniería de Sistemas  
**Unidad:** 2 — Hashing e índices basados en hash  
**Prerrequisito:** Bloque 1 — Arrays y Listas Enlazadas  
**Prerrequisito para:** Hash básico, Hashing estático

---

## ¿Para qué sirve este notebook?

Este material construye el puente entre las estructuras del Bloque 1 y las tablas *hash* que se estudiarán en clase. La idea central es simple: en lugar de buscar un elemento recorriendo toda una colección en O(n), una función *hash* permite **calcular directamente dónde debería estar** ese elemento, logrando búsquedas en O(1) en el caso promedio.

Al finalizar este notebook, se espera que pueda:

- Explicar qué es una función *hash* (*hash function*) y qué propiedades debe cumplir.
- Identificar qué es una colisión (*collision*) y por qué es inevitable.
- Implementar una tabla *hash* con *encadenamiento separado* (*separate chaining*).
- Implementar una tabla *hash* con *direccionamiento abierto* (*open addressing*) usando *probing* lineal.
- Comparar ambas estrategias de manejo de colisiones.

---

## Contenido

1. [La función hash](#1-funcion-hash)
   - 1.1 Propiedades de una buena función hash
   - 1.2 Funciones hash comunes
2. [Colisiones (*Collisions*)](#2-colisiones)
   - 2.1 ¿Por qué son inevitables?
3. [Encadenamiento separado (*Separate Chaining*)](#3-separate-chaining)
4. [Direccionamiento abierto (*Open Addressing*)](#4-open-addressing)
   - 4.1 Probing lineal (*Linear Probing*)
   - 4.2 Probing cuadrático (*Quadratic Probing*)
5. [Comparación de estrategias](#5-comparacion)
6. [Ejercicios propuestos](#6-ejercicios)


---
## 1. La función hash

### Repaso teórico

Una ***hash function*** (función hash) es una función que transforma una *clave* (*key*) de dominio arbitrario en un número entero dentro de un rango fijo `[0, N-1]`, donde `N` es el número de *buckets* (cubetas) de la tabla.

```
h : key → índice ∈ [0, N-1]
```

La función *hash* es el corazón de toda tabla *hash*: si la función es buena, las claves se distribuyen uniformemente y las búsquedas son O(1). Si es mala, muchas claves caen en el mismo *bucket* y el rendimiento se degrada a O(n).

### 1.1 Propiedades de una buena función hash

| Propiedad | Descripción |
|---|---|
| **Determinismo** | La misma clave siempre produce el mismo índice |
| **Uniformidad** | Las claves se distribuyen equitativamente entre los *buckets* |
| **Eficiencia** | El cálculo debe ser O(1), nunca O(n) |
| **Efecto avalancha** | Un pequeño cambio en la clave produce un índice muy diferente |

### 1.2 Funciones hash comunes

**División (*Division method*):**  
La más simple y directa. Usa el operador módulo:

```
h(k) = k % N
```

Funciona bien cuando `N` es un número primo. Si `N` es potencia de 2, solo usa los bits menos significativos de la clave, lo que puede generar agrupamiento (*clustering*).

**Multiplicación (*Multiplication method*):**  
Multiplica la clave por una constante `A` (típicamente la razón áurea `0.6180...`):

```
h(k) = floor(N × ((k × A) mod 1))
```

Menos sensible a la elección de `N`.

#### Ilustración — ¿Qué hace una función hash?

<div align="center">
  <img src="https://raw.githubusercontent.com/DS-UdeA/2026-1/refs/heads/main/clases/clase_06/notebooks/images/hash_mapping.png" alt="Hash mapping">
</div>

> *Fuente: Bhargava, A. (2016). Grokking Algorithms. Manning. Capítulo 5.*  
> Las claves ("milk", "apple", "avocado") entran por la función hash 
> y cada una sale hacia una posición específica del arreglo — siempre 
> la misma clave, siempre la misma posición.

In [1]:
# Hash functions — division and multiplication methods
import math

def division_hash(key: int, n: int) -> int:
    """
    Division method: h(k) = k % n
    Simple and effective when n is prime.

    Args:
        key (int): The integer key to hash.
        n   (int): Number of buckets.

    Returns:
        int: Bucket index in [0, n-1].
    """
    return key % n


def multiplication_hash(key: int, n: int) -> int:
    """
    Multiplication method: h(k) = floor(n * ((k * A) mod 1))
    Uses the golden ratio constant A ≈ 0.6180339887.

    Args:
        key (int): The integer key to hash.
        n   (int): Number of buckets.

    Returns:
        int: Bucket index in [0, n-1].
    """
    A = (math.sqrt(5) - 1) / 2  # Golden ratio ≈ 0.6180339887
    return math.floor(n * ((key * A) % 1))


def string_hash(key: str, n: int) -> int:
    """
    Polynomial hash for string keys.
    Uses the sum of (char_code * base^i) mod n.

    Args:
        key (str): The string key to hash.
        n   (int): Number of buckets.

    Returns:
        int: Bucket index in [0, n-1].
    """
    base = 31
    total = 0
    for i, char in enumerate(key):
        total += ord(char) * (base ** i)
    return total % n

#### Ejemplo 1 — Comparando las funciones hash

In [2]:
# Example 1: Comparing hash functions side by side
N = 7  # Prime number of buckets
keys = [5, 3, 8, 10, 9, 7, 14, 21, 36]

print(f"N = {N} buckets\n")
print(f"{'Key':>6} | {'Division h(k)=k%N':>20} | {'Multiplication':>16}")
print("-" * 50)
for k in keys:
    d = division_hash(k, N)
    m = multiplication_hash(k, N)
    print(f"{k:>6} | {d:>20} | {m:>16}")

N = 7 buckets

   Key |    Division h(k)=k%N |   Multiplication
--------------------------------------------------
     5 |                    5 |                0
     3 |                    3 |                5
     8 |                    1 |                6
    10 |                    3 |                1
     9 |                    2 |                3
     7 |                    0 |                2
    14 |                    0 |                4
    21 |                    0 |                6
    36 |                    1 |                1


#### Ejemplo 2 — Visualizando la distribución de claves

In [4]:
# Example 2: Visualizing key distribution across buckets
import random

N = 11  # Prime
random.seed(42)
sample_keys = [random.randint(0, 200) for _ in range(40)]

distribution = [0] * N
for k in sample_keys:
    distribution[division_hash(k, N)] += 1

print(f"Distribution of 40 random keys across {N} buckets:")
print()
for i, count in enumerate(distribution):
    bar = "█" * count
    print(f"  bucket[{i:2}]: {bar} ({count})")

Distribution of 40 random keys across 11 buckets:

  bucket[ 0]: ████ (4)
  bucket[ 1]: █████ (5)
  bucket[ 2]: █████ (5)
  bucket[ 3]: █ (1)
  bucket[ 4]: ████ (4)
  bucket[ 5]: █ (1)
  bucket[ 6]: ████ (4)
  bucket[ 7]: ████████ (8)
  bucket[ 8]: █████ (5)
  bucket[ 9]: ███ (3)
  bucket[10]:  (0)


---
## 2. Colisiones (*Collisions*)

### Repaso teórico

Una ***collision*** (colisión) ocurre cuando dos claves distintas producen el mismo índice *hash*:

```
k1 ≠ k2  pero  h(k1) = h(k2)
```

#### Ilustración — ¿Qué es una colisión?

<div align="center">
  <img src="https://raw.githubusercontent.com/DS-UdeA/2026-1/refs/heads/main/clases/clase_06/notebooks/images/collisions.png" alt="Collision">
</div>

> *Fuente: Bhargava, A. (2016). Grokking Algorithms. Manning. Capítulo 5.*  
> "APPLES" y "AVOCADOS" producen el mismo índice hash — ambas 
> apuntan a la posición 0.67. Eso es una colisión: dos claves 
> distintas, misma posición.

### 2.1 ¿Por qué son inevitables?

Por el ***Pigeonhole Principle*** (principio del palomar): si se tienen más claves que *buckets*, al menos dos claves deben caer en el mismo *bucket*. Incluso con pocas claves, la probabilidad de colisión crece rápidamente.

El ***Birthday Problem*** (paradoja del cumpleaños) ilustra esto: con solo 23 personas en una habitación, la probabilidad de que dos compartan cumpleaños supera el 50%. Análogamente, con N *buckets*, se espera la primera colisión después de insertar aproximadamente **√N** claves.

> 💡 **Conclusión:** Las colisiones no son un error de diseño. Son matemáticamente inevitables. El objetivo del diseño no es evitarlas, sino **manejarlas eficientemente**.

Existen dos grandes familias de estrategias para manejar colisiones:
1. **Encadenamiento separado** (*separate chaining*) — cada *bucket* almacena una lista de claves.
2. **Direccionamiento abierto** (*open addressing*) — todas las claves viven en la misma tabla; se busca otra posición libre.


In [5]:
# Demonstrating the Birthday Problem with hash collisions
import random

def first_collision_at(n_buckets: int, trials: int = 10_000) -> float:
    """
    Estimate the average number of insertions before the first collision
    in a table of n_buckets, using a Monte Carlo simulation.

    Args:
        n_buckets (int): Number of buckets in the hash table.
        trials    (int): Number of simulation runs.

    Returns:
        float: Average number of insertions before first collision.
    """
    total = 0
    for _ in range(trials):
        seen = set()
        count = 0
        while True:
            bucket = random.randint(0, n_buckets - 1)
            count += 1
            if bucket in seen:
                break
            seen.add(bucket)
        total += count
    return total / trials

print("Birthday problem applied to hash tables:")
print(f"{'N buckets':>12} | {'sqrt(N)':>10} | {'Avg. insertions before 1st collision':>38}")
print("-" * 68)
for n in [10, 50, 100, 365, 1000]:
    avg = first_collision_at(n, trials=5000)
    print(f"{n:>12} | {n**0.5:>10.1f} | {avg:>38.1f}")

Birthday problem applied to hash tables:
   N buckets |    sqrt(N) |   Avg. insertions before 1st collision
--------------------------------------------------------------------
          10 |        3.2 |                                    4.7
          50 |        7.1 |                                    9.5
         100 |       10.0 |                                   13.1
         365 |       19.1 |                                   24.3
        1000 |       31.6 |                                   40.1


---
## 3. Encadenamiento separado (*Separate Chaining*)

### Repaso teórico

En el *separate chaining*, cada posición de la tabla (*bucket*) almacena una **lista enlazada** de todos los pares clave-valor que colisionaron en ese índice.

```
tabla[h(k)] → [ (k1, v1) → (k2, v2) → None ]
```

**Operaciones:**
- **Inserción:** calcular `h(k)`, agregar al inicio de la lista en `tabla[h(k)]`. **O(1)**.
- **Búsqueda:** calcular `h(k)`, recorrer la lista en `tabla[h(k)]` hasta encontrar `k`. **O(1)** promedio, **O(n)** peor caso.
- **Eliminación:** calcular `h(k)`, eliminar `k` de la lista en `tabla[h(k)]`. **O(1)** promedio.

> 💡 El rendimiento promedio depende de cuántas claves por *bucket* haya en promedio. Ese número es el ***load factor*** (factor de carga) α = n/N, donde `n` es el número de claves insertadas y `N` el número de *buckets*.

#### Ilustración — Separate chaining en acción

<div align="center">
  <img src="https://raw.githubusercontent.com/DS-UdeA/2026-1/refs/heads/main/clases/clase_06/notebooks/images/separate_chaining.png" alt="Separate chaining diagram">
</div>

> *Fuente: Bhargava, A. (2016). Grokking Algorithms. Manning. Capítulo 5.*  
> "APPLES", "AVOCADOS" y "BANANAS" colisionan en el mismo bucket.  
> En lugar de descartar las colisiones, cada bucket encadena todos  
> los valores en una lista enlazada.

#### Diagrama UML — `HashTableChaining`

<div align="center">
  <img src="https://raw.githubusercontent.com/DS-UdeA/2026-1/refs/heads/main/clases/clase_06/notebooks/images/hash_table_chaining.png" alt="hash table chaining">
</div>

#### Documentación de la clase `HashTableChaining`

| Elemento | Descripción |
|---|---|
| **Clase** | `HashTableChaining` |
| **Propósito** | Tabla hash que resuelve colisiones con listas enlazadas por *bucket* |
| `_buckets` | Lista de listas; cada posición es un *bucket* con pares `(key, value)` |
| `_n_buckets` | Número total de *buckets* (N) |
| `_size` | Número de pares clave-valor almacenados actualmente |
| `put(key, value)` | Inserta o actualiza un par. **O(1)** promedio |
| `get(key)` | Retorna el valor asociado a `key`. Lanza `KeyError` si no existe. **O(1)** promedio |
| `delete(key)` | Elimina el par con la clave dada. **O(1)** promedio |
| `load_factor()` | Retorna α = size / n_buckets |
| `_hash(key)` | Función hash interna. Usa `hash()` nativo de Python combinado con módulo |


In [6]:
# Hash Table with Separate Chaining

class HashTableChaining:
    """
    A hash table that handles collisions using separate chaining.
    Each bucket holds a list of (key, value) pairs.
    """

    def __init__(self, n_buckets: int = 11):
        """
        Initialize the hash table.

        Args:
            n_buckets (int): Number of buckets. Prefer a prime number for better distribution.
        """
        self._n_buckets = n_buckets
        self._buckets = [[] for _ in range(n_buckets)]
        self._size = 0

    def _hash(self, key) -> int:
        """
        Internal hash function. O(1)
        Uses Python's built-in hash() combined with modulo.
        """
        return hash(key) % self._n_buckets

    def put(self, key, value) -> None:
        """
        Insert or update a key-value pair. O(1) average.

        If the key already exists, its value is updated.

        Args:
            key:   The search key.
            value: The associated value.
        """
        index = self._hash(key)
        bucket = self._buckets[index]
        for i, (k, v) in enumerate(bucket):
            if k == key:
                bucket[i] = (key, value)  # Update existing key
                return
        bucket.append((key, value))       # New key
        self._size += 1

    def get(self, key):
        """
        Return the value associated with key. O(1) average.

        Args:
            key: The key to look up.

        Returns:
            The value associated with key.

        Raises:
            KeyError: If key is not found.
        """
        index = self._hash(key)
        for k, v in self._buckets[index]:
            if k == key:
                return v
        raise KeyError(f"Key '{key}' not found.")

    def delete(self, key) -> bool:
        """
        Remove the entry with the given key. O(1) average.

        Args:
            key: The key to remove.

        Returns:
            True if removed, False if key was not found.
        """
        index = self._hash(key)
        bucket = self._buckets[index]
        for i, (k, v) in enumerate(bucket):
            if k == key:
                bucket.pop(i)
                self._size -= 1
                return True
        return False

    def load_factor(self) -> float:
        """Return α = size / n_buckets."""
        return self._size / self._n_buckets

    def __str__(self) -> str:
        lines = []
        for i, bucket in enumerate(self._buckets):
            if bucket:
                lines.append(f"  [{i:2}]: {bucket}")
        return "\n".join(lines) if lines else "  (empty table)"

#### Ejemplo 3 — Insertando claves y observando colisiones

In [7]:
# Example 3: Inserting keys and inspecting the bucket structure
ht = HashTableChaining(n_buckets=7)

entries = [(5, 'five'), (3, 'three'), (8, 'eight'),
           (10, 'ten'), (9, 'nine'), (7, 'seven'), (12, 'twelve')]

for key, value in entries:
    ht.put(key, value)
    print(f"  put({key:2}, '{value}') → bucket[{hash(key) % 7}], α = {ht.load_factor():.2f}")

print()
print("Table state:")
print(ht)

  put( 5, 'five') → bucket[5], α = 0.14
  put( 3, 'three') → bucket[3], α = 0.29
  put( 8, 'eight') → bucket[1], α = 0.43
  put(10, 'ten') → bucket[3], α = 0.57
  put( 9, 'nine') → bucket[2], α = 0.71
  put( 7, 'seven') → bucket[0], α = 0.86
  put(12, 'twelve') → bucket[5], α = 1.00

Table state:
  [ 0]: [(7, 'seven')]
  [ 1]: [(8, 'eight')]
  [ 2]: [(9, 'nine')]
  [ 3]: [(3, 'three'), (10, 'ten')]
  [ 5]: [(5, 'five'), (12, 'twelve')]


#### Ejemplo 4 — Búsqueda y eliminación

In [8]:
# Example 4: Search and delete operations
print("Search for key 9  :", ht.get(9))
print("Search for key 5  :", ht.get(5))

try:
    ht.get(99)
except KeyError as e:
    print("KeyError caught:", e)

print()
ht.delete(9)
print("After deleting key 9:")
print(ht)
print(f"\nSize: {ht._size}  |  Load factor α: {ht.load_factor():.2f}")

Search for key 9  : nine
Search for key 5  : five
KeyError caught: "Key '99' not found."

After deleting key 9:
  [ 0]: [(7, 'seven')]
  [ 1]: [(8, 'eight')]
  [ 3]: [(3, 'three'), (10, 'ten')]
  [ 5]: [(5, 'five'), (12, 'twelve')]

Size: 6  |  Load factor α: 0.86


#### Ejemplo 5 — Tabla hash con claves de tipo string

In [9]:
# Example 5: Using string keys (e.g., a simple phone book)
phone_book = HashTableChaining(n_buckets=13)

contacts = [
    ('Alice',   '300-111-2222'),
    ('Bob',     '301-333-4444'),
    ('Carlos',  '302-555-6666'),
    ('Diana',   '303-777-8888'),
    ('Eduardo', '304-999-0000'),
]

for name, number in contacts:
    phone_book.put(name, number)

print("Phone book state:")
print(phone_book)
print()
print("Alice's number  :", phone_book.get('Alice'))
print("Diana's number  :", phone_book.get('Diana'))

Phone book state:
  [ 2]: [('Eduardo', '304-999-0000')]
  [ 9]: [('Alice', '300-111-2222'), ('Diana', '303-777-8888')]
  [10]: [('Bob', '301-333-4444')]
  [11]: [('Carlos', '302-555-6666')]

Alice's number  : 300-111-2222
Diana's number  : 303-777-8888


---
## 4. Direccionamiento abierto (*Open Addressing*)

### Repaso teórico

En el *open addressing*, **todos los pares clave-valor se almacenan dentro de la misma tabla** (no hay listas externas). Cuando se produce una colisión en `h(k)`, se busca la siguiente posición disponible mediante una **secuencia de sondeo** (*probing sequence*).

Las ventajas sobre el *separate chaining* son:
- Mejor localidad de caché (*cache locality*): todos los datos están en un solo bloque contiguo de memoria.
- Sin *overhead* de punteros.

La desventaja principal es que el ***clustering*** (agrupamiento) degrada el rendimiento cuando el factor de carga es alto. Por esta razón, las tablas con *open addressing* deben mantenerse con α < 0.7.

### 4.1 Probing lineal (*Linear Probing*)

La secuencia de sondeo más simple: si `tabla[h(k)]` está ocupado, se prueba `h(k)+1`, luego `h(k)+2`, y así sucesivamente (en forma circular):

```
posición_i = (h(k) + i) % N
```

El problema del *linear probing* es el **agrupamiento primario** (*primary clustering*): las claves tienden a amontonarse en bloques contiguos, lo que hace que las inserciones futuras en esa zona sean más lentas.

#### Ilustración — Primary clustering en linear probing

<div align="center">
  <img src="https://raw.githubusercontent.com/DS-UdeA/2026-1/refs/heads/main/clases/clase_06/notebooks/images/linear_probing.png" alt="Linear probing clustering">
</div>



> *Fuente: Goodrich, Tamassia & Goldwasser (2013). Data Structures  
> and Algorithms in Python. Fig. 10.7.*  
> La clave 15 debe sondear 4 veces antes de encontrar un slot libre.  
> Las posiciones 4–7 forman un cluster — ese es el costo real  
> del agrupamiento primario en linear probing.

### 4.2 Probing cuadrático (*Quadratic Probing*)

Reduce el agrupamiento primario usando una función cuadrática para la secuencia de sondeo:

```
posición_i = (h(k) + i²) % N
```

Distribuye mejor los elementos, pero puede sufrir **agrupamiento secundario** (*secondary clustering*): dos claves con el mismo *hash* inicial siguen exactamente la misma secuencia de sondeo.


#### Diagrama UML — `HashTableOpenAddressing`

<div align="center">
  <img src="https://raw.githubusercontent.com/DS-UdeA/2026-1/refs/heads/main/clases/clase_06/notebooks/images/hash_table_open_addressing.png" alt="Hast Table Open Addressing">
</div>

#### Documentación de la clase `HashTableOpenAddressing`

| Elemento | Descripción |
|---|---|
| **Clase** | `HashTableOpenAddressing` |
| **Propósito** | Tabla hash que resuelve colisiones buscando la siguiente posición libre dentro de la misma tabla |
| `_keys` | Array interno que almacena las claves |
| `_values` | Array interno que almacena los valores |
| `_n_buckets` | Número de posiciones en la tabla |
| `_DELETED` | Marcador especial (*sentinel*) para posiciones eliminadas lógicamente |
| `_probe_type` | `'linear'` o `'quadratic'` |
| `put(key, value)` | Inserta o actualiza. **O(1)** promedio |
| `get(key)` | Retorna el valor asociado a `key`. **O(1)** promedio |
| `delete(key)` | Marca la posición como eliminada (*lazy deletion*). **O(1)** promedio |
| `_probe(index, i)` | Calcula la posición del i-ésimo sondeo para el índice inicial dado |


In [10]:
# Hash Table with Open Addressing (Linear and Quadratic Probing)

class HashTableOpenAddressing:
    """
    A hash table that resolves collisions using open addressing.
    Supports both linear probing and quadratic probing.

    Important: keep load factor α < 0.7 to maintain O(1) average performance.
    """

    _DELETED = object()  # Sentinel for lazily deleted slots

    def __init__(self, n_buckets: int = 11, probe_type: str = 'linear'):
        """
        Initialize the table.

        Args:
            n_buckets  (int): Number of slots. Prefer a prime number.
            probe_type (str): 'linear' or 'quadratic'.
        """
        if probe_type not in ('linear', 'quadratic'):
            raise ValueError("probe_type must be 'linear' or 'quadratic'.")
        self._n_buckets = n_buckets
        self._keys   = [None] * n_buckets
        self._values = [None] * n_buckets
        self._size   = 0
        self._probe_type = probe_type

    def _hash(self, key) -> int:
        """Primary hash function."""
        return hash(key) % self._n_buckets

    def _probe(self, index: int, i: int) -> int:
        """
        Calculate the i-th probe position starting from index.

        Args:
            index (int): Initial hash position h(k).
            i     (int): Probe step number (0 = first attempt, no displacement).

        Returns:
            int: The next slot to check.
        """
        if self._probe_type == 'linear':
            return (index + i) % self._n_buckets
        else:  # quadratic
            return (index + i * i) % self._n_buckets

    def put(self, key, value) -> None:
        """
        Insert or update a key-value pair. O(1) average.
        Raises RuntimeError if the table is full.
        """
        if self._size >= self._n_buckets:
            raise RuntimeError("Hash table is full.")
        index = self._hash(key)
        first_deleted = None
        for i in range(self._n_buckets):
            pos = self._probe(index, i)
            if self._keys[pos] is self._DELETED:
                if first_deleted is None:
                    first_deleted = pos
            elif self._keys[pos] is None:
                insert_at = first_deleted if first_deleted is not None else pos
                self._keys[insert_at]   = key
                self._values[insert_at] = value
                self._size += 1
                return
            elif self._keys[pos] == key:
                self._values[pos] = value  # Update
                return
        if first_deleted is not None:
            self._keys[first_deleted]   = key
            self._values[first_deleted] = value
            self._size += 1

    def get(self, key):
        """
        Return the value for key. O(1) average.
        Raises KeyError if not found.
        """
        index = self._hash(key)
        for i in range(self._n_buckets):
            pos = self._probe(index, i)
            if self._keys[pos] is None:
                break
            if self._keys[pos] == key:
                return self._values[pos]
        raise KeyError(f"Key '{key}' not found.")

    def delete(self, key) -> bool:
        """
        Lazily delete a key (marks slot as DELETED). O(1) average.

        Lazy deletion is required: physically removing the slot would
        break probing chains for keys inserted after this one.
        """
        index = self._hash(key)
        for i in range(self._n_buckets):
            pos = self._probe(index, i)
            if self._keys[pos] is None:
                return False
            if self._keys[pos] == key:
                self._keys[pos]   = self._DELETED
                self._values[pos] = None
                self._size -= 1
                return True
        return False

    def load_factor(self) -> float:
        """Return α = size / n_buckets."""
        return self._size / self._n_buckets

    def __str__(self) -> str:
        lines = []
        for i in range(self._n_buckets):
            if self._keys[i] is None:
                lines.append(f"  [{i:2}]: —")
            elif self._keys[i] is self._DELETED:
                lines.append(f"  [{i:2}]: <DELETED>")
            else:
                lines.append(f"  [{i:2}]: ({self._keys[i]}, {self._values[i]})")
        return "\n".join(lines)

#### Ejemplo 6 — *Linear probing*: observando el agrupamiento (*clustering*)

In [11]:
# Example 6: Linear probing — watching primary clustering form
lp = HashTableOpenAddressing(n_buckets=11, probe_type='linear')

keys = [5, 16, 27, 38, 9, 20]
# Note: 5, 16, 27, 38 all hash to index 5 in a table of 11 (5%11=5, 16%11=5, etc.)

for k in keys:
    lp.put(k, f'val_{k}')
    print(f"  put({k:2}) → primary slot = {k % 11}, α = {lp.load_factor():.2f}")

print()
print("Table state (notice the cluster forming around slot 5):")
print(lp)

  put( 5) → primary slot = 5, α = 0.09
  put(16) → primary slot = 5, α = 0.18
  put(27) → primary slot = 5, α = 0.27
  put(38) → primary slot = 5, α = 0.36
  put( 9) → primary slot = 9, α = 0.45
  put(20) → primary slot = 9, α = 0.55

Table state (notice the cluster forming around slot 5):
  [ 0]: —
  [ 1]: —
  [ 2]: —
  [ 3]: —
  [ 4]: —
  [ 5]: (5, val_5)
  [ 6]: (16, val_16)
  [ 7]: (27, val_27)
  [ 8]: (38, val_38)
  [ 9]: (9, val_9)
  [10]: (20, val_20)


#### Ejemplo 7 — Comparando *linear* vs *quadratic probing*

In [12]:
# Example 7: Comparing probe sequences for the same set of colliding keys
colliding_keys = [5, 16, 27, 38]  # All hash to slot 5 in a table of 11

linear_ht = HashTableOpenAddressing(n_buckets=11, probe_type='linear')
quad_ht   = HashTableOpenAddressing(n_buckets=11, probe_type='quadratic')

for k in colliding_keys:
    linear_ht.put(k, f'v{k}')
    quad_ht.put(k, f'v{k}')

print("Linear probing result:")
print(linear_ht)
print()
print("Quadratic probing result:")
print(quad_ht)

Linear probing result:
  [ 0]: —
  [ 1]: —
  [ 2]: —
  [ 3]: —
  [ 4]: —
  [ 5]: (5, v5)
  [ 6]: (16, v16)
  [ 7]: (27, v27)
  [ 8]: (38, v38)
  [ 9]: —
  [10]: —

Quadratic probing result:
  [ 0]: —
  [ 1]: —
  [ 2]: —
  [ 3]: (38, v38)
  [ 4]: —
  [ 5]: (5, v5)
  [ 6]: (16, v16)
  [ 7]: —
  [ 8]: —
  [ 9]: (27, v27)
  [10]: —


#### Ejemplo 8 — Por qué la eliminación debe ser *lazy* (*lazy deletion*)

Este ejemplo muestra qué ocurriría si se eliminara una entrada físicamente en lugar de marcarla con un *sentinel*:

In [13]:
# Example 8: Why lazy deletion is necessary in open addressing

# Insert three keys that collide and form a probe chain: 5, 16, 27
ht_lazy = HashTableOpenAddressing(n_buckets=11, probe_type='linear')
ht_lazy.put(5,  'five')
ht_lazy.put(16, 'sixteen')  # Collides with 5 → goes to slot 6
ht_lazy.put(27, 'twenty-seven')  # Collides → goes to slot 7

print("Before deletion:")
print(ht_lazy)
print()

# Delete key 16 using lazy deletion (marks slot 6 as DELETED)
ht_lazy.delete(16)
print("After lazy-deleting key 16 (slot 6 marked as DELETED):")
print(ht_lazy)
print()

# Now search for key 27:
# The probe chain is: slot 5 (occupied by 5) → slot 6 (DELETED, keep going) → slot 7 (found!)
# If slot 6 were truly empty (None), the search would STOP there and incorrectly report 27 as not found.
result = ht_lazy.get(27)
print(f"Search for key 27 after deletion of 16: '{result}' ✓")

Before deletion:
  [ 0]: —
  [ 1]: —
  [ 2]: —
  [ 3]: —
  [ 4]: —
  [ 5]: (5, five)
  [ 6]: (16, sixteen)
  [ 7]: (27, twenty-seven)
  [ 8]: —
  [ 9]: —
  [10]: —

After lazy-deleting key 16 (slot 6 marked as DELETED):
  [ 0]: —
  [ 1]: —
  [ 2]: —
  [ 3]: —
  [ 4]: —
  [ 5]: (5, five)
  [ 6]: <DELETED>
  [ 7]: (27, twenty-seven)
  [ 8]: —
  [ 9]: —
  [10]: —

Search for key 27 after deletion of 16: 'twenty-seven' ✓


---
## 5. Comparación de estrategias

### Resumen de complejidad — Notación O

| Operación | Separate Chaining | Linear Probing | Quadratic Probing |
|---|:---:|:---:|:---:|
| Búsqueda (promedio) | **O(1 + α)** | **O(1/(1-α))** | **O(1/(1-α))** |
| Búsqueda (peor caso) | O(n) | O(n) | O(n) |
| Inserción (promedio) | **O(1)** | **O(1/(1-α))** | **O(1/(1-α))** |
| Eliminación | O(1) promedio | O(1) lazy | O(1) lazy |
| Factor de carga recomendado | α ≤ 1.0 | **α < 0.7** | **α < 0.7** |
| Memoria extra | Punteros de lista | Ninguna | Ninguna |
| Localidad de caché | Baja | **Alta** | Media |
| Agrupamiento | No aplica | Primario | Secundario |

<br>

> 📌 **Nota:** α (alpha) es el ***load factor*** (factor de carga) = n / N, donde `n` es el número de entradas y `N` el número de *buckets*. El factor de carga es el indicador más importante del rendimiento de una tabla *hash*. Este concepto se profundiza en el **Bloque 3**.

### ¿Cuándo usar cada estrategia?

| Situación | Estrategia recomendada |
|---|---|
| No se conoce el número de claves de antemano | *Separate chaining* (crece con facilidad) |
| Se necesita máximo rendimiento de caché | *Linear probing* con α < 0.5 |
| Se quiere evitar listas externas | Cualquier *open addressing* |
| Eliminaciones frecuentes | *Separate chaining* (sin *lazy deletion*) |
| Hashing en disco (bases de datos) | Se usan variantes especializadas — **ver Clases 6 y 7** |


In [15]:
# Empirical comparison: measuring average probe steps as load factor increases

import random

def measure_avg_probes(n_buckets: int, n_insertions: int, probe_type: str) -> float:
    """
    Measure average number of probe steps needed per insertion.

    Args:
        n_buckets    (int): Number of buckets.
        n_insertions (int): Number of keys to insert.
        probe_type   (str): 'linear' or 'quadratic'.

    Returns:
        float: Average probes per insertion.
    """
    keys_array = [None] * n_buckets
    total_probes = 0
    inserted = 0
    keys = random.sample(range(n_buckets * 10), n_insertions)
    for key in keys:
        index = key % n_buckets
        for i in range(n_buckets):
            if probe_type == 'linear':
                pos = (index + i) % n_buckets
            else:
                pos = (index + i * i) % n_buckets
            total_probes += 1
            if keys_array[pos] is None:
                keys_array[pos] = key
                break
        inserted += 1
    return total_probes / inserted

N = 97  # Prime
print(f"N = {N} buckets")
print(f"{'α':>6} | {'n keys':>8} | {'Linear avg probes':>20} | {'Quadratic avg probes':>22}")
print("-" * 65)

for alpha in [0.1, 0.3, 0.5, 0.6, 0.7, 0.8, 0.9]:
    n = int(N * alpha)
    lp_avg = measure_avg_probes(N, n, 'linear')
    qp_avg = measure_avg_probes(N, n, 'quadratic')
    print(f"{alpha:>6.1f} | {n:>8} | {lp_avg:>20.2f} | {qp_avg:>22.2f}")

N = 97 buckets
     α |   n keys |    Linear avg probes |   Quadratic avg probes
-----------------------------------------------------------------
   0.1 |        9 |                 1.11 |                   1.11
   0.3 |       29 |                 1.17 |                   1.21
   0.5 |       48 |                 1.23 |                   1.38
   0.6 |       58 |                 1.40 |                   1.33
   0.7 |       67 |                 1.58 |                   1.69
   0.8 |       77 |                 2.88 |                   1.94
   0.9 |       87 |                 2.87 |                   2.76


---
## 6. Ejercicios propuestos

Se recomienda resolver los ejercicios en orden antes de la siguiente clase.

In [ ]:
# Exercise 1
# Given the hash function h(k) = k % 7 and the keys:
#   [10, 24, 45, 17, 3, 38, 31]
#
# a) Manually compute the bucket index for each key.
# b) Build a HashTableChaining with N=7 and insert all keys.
# c) Identify which keys collide and in which bucket.
# d) What is the load factor after all insertions?

# Write your solution here


In [ ]:
# Exercise 2
# Insert the keys [5, 16, 27, 38, 49] into a HashTableOpenAddressing
# with N=11 using linear probing.
#
# a) Trace step by step where each key lands (which slot).
# b) Then delete key 16 using lazy deletion.
# c) Insert key 60 after the deletion. Where does it land?
# d) Can you still find key 27 after the deletion? Why?

# Write your solution here


In [ ]:
# Exercise 3 (challenge)
# Implement a method resize(new_n_buckets) for HashTableChaining that:
# - Creates a new table with new_n_buckets
# - Reinserts all existing (key, value) pairs into the new table
# - Replaces the internal state of the object
#
# Then, implement an auto_resize() that calls resize() automatically
# when load_factor() exceeds 0.75, doubling the number of buckets.
#
# Hint: The new number of buckets should ideally be a prime number.
#       You may use a helper function to find the next prime.

# Write your solution here


## Referencias de este bloque

### Referencias principales

- **Goodrich, Tamassia & Goldwasser (2013).** *Data Structures and Algorithms in Python*. Wiley. **Capítulo 10 — Maps, Hash Tables & Skip Lists.**
- **Bhargava, A. (2016).** *Grokking Algorithms*. Manning. **Capítulo 5 — Hash Tables.**
- **Ramakrishnan & Gehrke (2003).** *Database Management Systems* (3rd ed.). McGraw-Hill. **Capítulo de Hashing.**
- Visualizador interactivo: [USFCA — Open Hash](https://www.cs.usfca.edu/~galles/visualization/OpenHash.html)
- Visualizador interactivo: [VisuAlgo — Hash Table](https://visualgo.net/en/hashtable)

---

### Conexión con Silberschatz — *Database System Concepts* (7th ed.)

| Sección | Tema | Conecta con |
|---|---|---|
| Cap. 14 — §14.4 | Static Hashing | Función hash `h(k) = k % N` en índices de disco |
| Cap. 14 — §14.4.1 | Hash Indexes | *Separate chaining* aplicado a *buckets* en disco |
| Cap. 14 — §14.4.2 | Handling of Bucket Overflows | Colisiones y *overflow chaining* |

<br>

> 💡 **Lectura recomendada:** leer §14.4 completo después de completar este notebook, antes de pasar al Bloque 3.

---

> 🤖 **AI Disclosure:** 
> This document was created with the assistance of Artificial Intelligence language models. The content has been reviewed, edited, and validated by a human author to ensure accuracy and quality.